# Panzer -- Peticiones en paralelo (bulk)

Panzer permite lanzar multiples peticiones a la API de Binance
**simultaneamente** usando un pool de threads, con control total
del peso consumido.

El flujo interno es:

1. **Estimar pesos** de todas las peticiones antes de lanzarlas.
2. **Agrupar en lotes** que quepan en la ventana de rate limiting.
3. **Reservar** todo el peso del lote de golpe.
4. **Lanzar** las peticiones en paralelo (sin competir por el lock).
5. **Recoger** resultados y sincronizar con las cabeceras del servidor.

Esto aprovecha el margen libre de peso para maximizar el throughput
en llamadas pesadas (trades, depth, klines de muchos simbolos).

## 1. Crear cliente y sincronizar reloj

In [1]:
from panzer import BinancePublicClient

client = BinancePublicClient(market="spot", safety_ratio=0.9)
client.ensure_time_offset_ready(min_samples=3)

print(f"Mercado:          {client.market}")
print(f"Max peso/min:     {client.limiter.max_per_minute}")
print(f"Limite efectivo:  {client.limiter.effective_limit}")
print(f"Peso disponible:  {client.limiter.remaining}")

2026-03-06 19:33:42     INFO [panzer.binance.public.spot] Inicializando BinancePublicClient(market=spot)
2026-03-06 19:33:43     INFO [panzer.binance.public.spot] Rate limiter inicializado: max_per_minute=6000 safety_ratio=0.90


Mercado:          spot
Max peso/min:     6000
Limite efectivo:  5400
Peso disponible:  5377


## 2. bulk_trades -- trades de multiples simbolos

Cada llamada a `/trades` con limit=500 pesa 25. Pedir 10 simbolos en
secuencial cuesta ~2.5s (latencia de red). En paralelo, ~0.3s.

In [2]:
import time

symbols = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "XRPUSDT", "BNBUSDT",
           "ADAUSDT", "DOGEUSDT", "AVAXUSDT", "DOTUSDT", "LINKUSDT"]

# --- Secuencial ---
t0 = time.time()
seq = {}
for s in symbols:
    seq[s] = client.trades(s, limit=100)
t_seq = time.time() - t0

# --- Paralelo ---
t0 = time.time()
par = client.bulk_trades(symbols, limit=100, max_workers=10)
t_par = time.time() - t0

print(f"Secuencial: {t_seq:.2f}s")
print(f"Paralelo:   {t_par:.2f}s")
print(f"Speedup:    {t_seq / t_par:.1f}x")
print()
for s in symbols:
    print(f"  {s:<10s}  seq={len(seq[s]):>3d} trades   par={len(par[s]):>3d} trades")

2026-03-06 19:33:47     INFO [panzer.binance.public.spot] parallel_get: 10 jobs, 1 lotes, used_local=523


Secuencial: 2.59s
Paralelo:   0.38s
Speedup:    6.8x

  BTCUSDT     seq=100 trades   par=100 trades
  ETHUSDT     seq=100 trades   par=100 trades
  SOLUSDT     seq=100 trades   par=100 trades
  XRPUSDT     seq=100 trades   par=100 trades
  BNBUSDT     seq=100 trades   par=100 trades
  ADAUSDT     seq=100 trades   par=100 trades
  DOGEUSDT    seq=100 trades   par=100 trades
  AVAXUSDT    seq=100 trades   par=100 trades
  DOTUSDT     seq=100 trades   par=100 trades
  LINKUSDT    seq=100 trades   par=100 trades


## 3. bulk_klines -- velas de multiples simbolos

Mismo intervalo y limite para todos los simbolos. Peso de klines = 2 por
peticion (muy ligero), asi que el cuello de botella es puramente la latencia.

In [3]:
t0 = time.time()
all_klines = client.bulk_klines(symbols, "1h", limit=24, max_workers=10)
elapsed = time.time() - t0

print(f"Obtenidas klines 1h (24 velas) de {len(symbols)} simbolos en {elapsed:.2f}s\n")
for sym, klines in all_klines.items():
    last_close = klines[-1][4]
    print(f"  {sym:<10s}  {len(klines)} velas  ultimo close={last_close}")

2026-03-06 19:33:47     INFO [panzer.binance.public.spot] parallel_get: 10 jobs, 1 lotes, used_local=543


Obtenidas klines 1h (24 velas) de 10 simbolos en 0.38s

  BTCUSDT     24 velas  ultimo close=68413.28000000
  ETHUSDT     24 velas  ultimo close=1981.17000000
  SOLUSDT     24 velas  ultimo close=85.08000000
  XRPUSDT     24 velas  ultimo close=1.35910000
  BNBUSDT     24 velas  ultimo close=631.27000000
  ADAUSDT     24 velas  ultimo close=0.25880000
  DOGEUSDT    24 velas  ultimo close=0.09065000
  AVAXUSDT    24 velas  ultimo close=8.98000000
  DOTUSDT     24 velas  ultimo close=1.48200000
  LINKUSDT    24 velas  ultimo close=8.80000000


## 4. bulk_depth -- order books en paralelo

depth con limit=100 pesa 5. Con limit=500 pesa 25. Aqui pedimos 10
order books de golpe.

In [4]:
t0 = time.time()
all_books = client.bulk_depth(symbols, limit=100, max_workers=10)
elapsed = time.time() - t0

print(f"Obtenidos {len(symbols)} order books en {elapsed:.2f}s\n")
for sym, book in all_books.items():
    best_bid = book["bids"][0][0]
    best_ask = book["asks"][0][0]
    spread = float(best_ask) - float(best_bid)
    print(f"  {sym:<10s}  bid={best_bid:>14s}  ask={best_ask:>14s}  spread={spread:.4f}")

2026-03-06 19:33:48     INFO [panzer.binance.public.spot] parallel_get: 10 jobs, 1 lotes, used_local=593


Obtenidos 10 order books en 0.37s

  BTCUSDT     bid=68413.27000000  ask=68413.28000000  spread=0.0100
  ETHUSDT     bid= 1981.23000000  ask= 1981.24000000  spread=0.0100
  SOLUSDT     bid=   85.07000000  ask=   85.08000000  spread=0.0100
  XRPUSDT     bid=    1.35910000  ask=    1.35920000  spread=0.0001
  BNBUSDT     bid=  631.26000000  ask=  631.27000000  spread=0.0100
  ADAUSDT     bid=    0.25880000  ask=    0.25890000  spread=0.0001
  DOGEUSDT    bid=    0.09064000  ask=    0.09065000  spread=0.0000
  AVAXUSDT    bid=    8.97000000  ask=    8.98000000  spread=0.0100
  DOTUSDT     bid=    1.48100000  ask=    1.48200000  spread=0.0010
  LINKUSDT    bid=    8.79000000  ask=    8.80000000  spread=0.0100


## 5. bulk_agg_trades -- trades agregados en paralelo

In [5]:
t0 = time.time()
all_agg = client.bulk_agg_trades(symbols[:5], limit=200, max_workers=5)
elapsed = time.time() - t0

print(f"Obtenidos aggTrades de {len(all_agg)} simbolos en {elapsed:.2f}s\n")
for sym, agg in all_agg.items():
    print(f"  {sym:<10s}  {len(agg)} aggTrades  ultimo precio={agg[-1]['p']}")

2026-03-06 19:33:48     INFO [panzer.binance.public.spot] parallel_get: 5 jobs, 1 lotes, used_local=613


Obtenidos aggTrades de 5 simbolos en 0.33s

  BTCUSDT     200 aggTrades  ultimo precio=68413.28000000
  ETHUSDT     200 aggTrades  ultimo precio=1981.23000000
  SOLUSDT     200 aggTrades  ultimo precio=85.08000000
  XRPUSDT     200 aggTrades  ultimo precio=1.35910000
  BNBUSDT     200 aggTrades  ultimo precio=631.27000000


## 6. parallel_get -- API generica

`parallel_get` es el metodo de bajo nivel que usan los `bulk_*`.
Acepta una lista de `(endpoint, params)` arbitrarios, incluso mezclando
endpoints distintos.

In [6]:
# Mezclar endpoints distintos en una sola llamada paralela
jobs = [
    ("/api/v3/klines", {"symbol": "BTCUSDT", "interval": "1d", "limit": 7}),
    ("/api/v3/trades", {"symbol": "ETHUSDT", "limit": 5}),
    ("/api/v3/depth",  {"symbol": "SOLUSDT", "limit": 20}),
    ("/api/v3/ticker/24hr", {"symbol": "XRPUSDT"}),
]

t0 = time.time()
results = client.parallel_get(jobs, max_workers=4)
elapsed = time.time() - t0

print(f"4 peticiones mixtas en {elapsed:.2f}s\n")

# Resultado 0: klines BTCUSDT 1d
print(f"BTCUSDT klines 1d: {len(results[0])} velas")

# Resultado 1: trades ETHUSDT
print(f"ETHUSDT trades:    {len(results[1])} trades")

# Resultado 2: depth SOLUSDT
print(f"SOLUSDT depth:     {len(results[2]['bids'])} niveles bid")

# Resultado 3: ticker 24h XRPUSDT
print(f"XRPUSDT 24h:       precio={results[3]['lastPrice']}  cambio={results[3]['priceChangePercent']}%")

2026-03-06 19:33:49     INFO [panzer.binance.public.spot] parallel_get: 4 jobs, 1 lotes, used_local=647


4 peticiones mixtas en 0.31s

BTCUSDT klines 1d: 7 velas
ETHUSDT trades:    5 trades
SOLUSDT depth:     20 niveles bid
XRPUSDT 24h:       precio=1.35910000  cambio=-3.315%


## 7. Control de peso -- inspeccion antes y despues

Las propiedades `effective_limit` y `remaining` del limiter permiten
saber exactamente cuanto margen hay antes de lanzar un lote.

In [7]:
from panzer.exchanges.binance.weights import get_weight

lim = client.limiter

print("=== Estado del limiter ===")
print(f"  Limite efectivo:  {lim.effective_limit}")
print(f"  Peso usado:       {lim.used_local}")
print(f"  Peso servidor:    {lim.last_server_used}")
print(f"  Restante:         {lim.remaining}")

# Cuanto costaria pedir trades de 50 simbolos?
weight_per_trade = get_weight("spot", "/api/v3/trades", {"limit": 500})
total_50 = weight_per_trade * 50
print(f"\n=== Estimacion: 50 x trades(limit=500) ===")
print(f"  Peso por llamada:  {weight_per_trade}")
print(f"  Peso total:        {total_50}")
print(f"  Cabe en ventana?   {'SI' if total_50 <= lim.remaining else 'NO, se partira en lotes'}")
print(f"  Lotes necesarios:  {-(-total_50 // lim.effective_limit)}")

=== Estado del limiter ===
  Limite efectivo:  5400
  Peso usado:       647
  Peso servidor:    622
  Restante:         4753

=== Estimacion: 50 x trades(limit=500) ===
  Peso por llamada:  25
  Peso total:        1250
  Cabe en ventana?   SI
  Lotes necesarios:  1


## 8. Batching automatico -- cuando el peso excede la ventana

Si el peso total de las peticiones supera el limite efectivo, `parallel_get`
las parte automaticamente en lotes. Cada lote reserva su peso y lanza en
paralelo. Entre lotes, el limiter duerme si hace falta.

Ejemplo: 100 llamadas a trades (peso 25 cada una) = 2500 de peso.
Con limite efectivo 5400, cabe en un solo lote. Pero si ya hemos
consumido 4000, el primer lote sera de ~56 llamadas y el segundo
esperara a la siguiente ventana.

In [8]:
# Pedir klines de 30 simbolos -- el limiter partira en lotes si necesario
big_list = [
    "BTCUSDT", "ETHUSDT", "BNBUSDT", "SOLUSDT", "XRPUSDT",
    "DOGEUSDT", "ADAUSDT", "AVAXUSDT", "DOTUSDT", "LINKUSDT",
    "MATICUSDT", "UNIUSDT", "ATOMUSDT", "LTCUSDT", "ETCUSDT",
    "XLMUSDT", "NEARUSDT", "APTUSDT", "FILUSDT", "ALGOUSDT",
    "VETUSDT", "ICPUSDT", "RUNEUSDT", "AAVEUSDT", "GRTUSDT",
    "FTMUSDT", "SANDUSDT", "MANAUSDT", "AXSUSDT", "THETAUSDT",
]

peso_klines = get_weight("spot", "/api/v3/klines", {"limit": 100})
print(f"Peso por klines(limit=100): {peso_klines}")
print(f"Peso total estimado: {peso_klines * len(big_list)}")
print(f"Restante en ventana: {client.limiter.remaining}")
print()

t0 = time.time()
result = client.bulk_klines(big_list, "15m", limit=100, max_workers=15)
elapsed = time.time() - t0

print(f"Obtenidas klines de {len(result)} simbolos en {elapsed:.2f}s")
print(f"Peso usado tras la operacion: {client.limiter.used_local}")

Peso por klines(limit=100): 2
Peso total estimado: 60
Restante en ventana: 4753



2026-03-06 19:33:50     INFO [panzer.binance.public.spot] parallel_get: 30 jobs, 1 lotes, used_local=707


Obtenidas klines de 30 simbolos en 0.81s
Peso usado tras la operacion: 707


## 9. Funciona igual con Futures

Los metodos `bulk_*` funcionan con cualquier mercado. Solo cambia
el parametro `market` del cliente.

In [9]:
futures = BinancePublicClient(market="um", safety_ratio=0.9)

futures_symbols = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "XRPUSDT", "BNBUSDT"]

t0 = time.time()
futures_trades = futures.bulk_trades(futures_symbols, limit=100, max_workers=5)
elapsed = time.time() - t0

print(f"Futures USDT-M: {len(futures_symbols)} simbolos en {elapsed:.2f}s")
print(f"Limite efectivo futures: {futures.limiter.effective_limit} (vs {client.limiter.effective_limit} spot)\n")
for sym, trades in futures_trades.items():
    print(f"  {sym:<10s}  {len(trades)} trades  ultimo precio={trades[-1]['price']}")

2026-03-06 19:33:50     INFO [panzer.binance.public.um] Inicializando BinancePublicClient(market=um)
2026-03-06 19:33:50     INFO [panzer.binance.public.um] Rate limiter inicializado: max_per_minute=2400 safety_ratio=0.90
2026-03-06 19:33:50     INFO [panzer.binance.public.um] parallel_get: 5 jobs, 1 lotes, used_local=26


Futures USDT-M: 5 simbolos en 0.33s
Limite efectivo futures: 2160 (vs 5400 spot)

  BTCUSDT     100 trades  ultimo precio=68383.10
  ETHUSDT     100 trades  ultimo precio=1980.29
  SOLUSDT     100 trades  ultimo precio=85.0100
  XRPUSDT     100 trades  ultimo precio=1.3579
  BNBUSDT     100 trades  ultimo precio=631.100


## 10. Manejo de errores

Si alguna peticion del lote falla (ej: simbolo invalido), se lanza
`BinanceAPIException` con el error de la primera peticion fallida.
Las demas peticiones del lote se completan igualmente.

In [10]:
from panzer.errors import BinanceAPIException

try:
    # "INVALIDO" no existe en Binance
    client.bulk_trades(["BTCUSDT", "INVALIDO", "ETHUSDT"], limit=5)
except BinanceAPIException as e:
    print(f"Error capturado:")
    print(f"  HTTP status:  {e.status_code}")
    print(f"  Binance code: {e.error_payload.code}")
    print(f"  Mensaje:      {e.error_payload.msg}")

2026-03-06 19:33:51    ERROR [panzer.errors] BinanceAPIException raised: status=400 method=GET url=https://api.binance.com/api/v3/trades?symbol=INVALIDO&limit=5 code=-1121 msg=Invalid symbol.


Error capturado:
  HTTP status:  400
  Binance code: -1121
  Mensaje:      Invalid symbol.


## 11. Resumen

| Metodo | Descripcion | Peso tipico |
|---|---|---|
| `bulk_trades(symbols)` | Trades recientes de N simbolos | 25 x N |
| `bulk_klines(symbols, interval)` | Klines de N simbolos | 2 x N |
| `bulk_depth(symbols)` | Order books de N simbolos | 5-250 x N |
| `bulk_agg_trades(symbols)` | AggTrades de N simbolos | 4 x N |
| `parallel_get(jobs)` | Peticiones arbitrarias mezcladas | variable |

Todos los metodos:
- **No rompen** la API existente -- los metodos secuenciales siguen igual.
- **Pre-reservan** peso antes de lanzar las peticiones.
- **Parten en lotes** si el peso total excede la ventana.
- **Duermen automaticamente** entre lotes si hace falta.
- Devuelven un `dict[symbol, data]` (bulk) o `list[data]` (parallel_get).